In [2]:
import requests
from bs4 import BeautifulSoup
 
headers = {'User-Agent': 'data-eng-class-crawler/1.0 (learning purpose)'}
res = requests.get('https://quotes.toscrape.com/', headers=headers, timeout=10)
print('status:', res.status_code)
 
soup = BeautifulSoup(res.text, 'html.parser')
for q in soup.select('div.quote'):
    text = q.select_one('span.text').get_text(strip=True)
    author = q.select_one('small.author').get_text(strip=True)
    tags = [t.get_text(strip=True) for t in q.select('a.tag')]
    print(author, '|', text[:30], '|', tags)

status: 200
Albert Einstein | “The world as we have created  | ['change', 'deep-thoughts', 'thinking', 'world']
J.K. Rowling | “It is our choices, Harry, tha | ['abilities', 'choices']
Albert Einstein | “There are only two ways to li | ['inspirational', 'life', 'live', 'miracle', 'miracles']
Jane Austen | “The person, be it gentleman o | ['aliteracy', 'books', 'classic', 'humor']
Marilyn Monroe | “Imperfection is beauty, madne | ['be-yourself', 'inspirational']
Albert Einstein | “Try not to become a man of su | ['adulthood', 'success', 'value']
André Gide | “It is better to be hated for  | ['life', 'love']
Thomas A. Edison | “I have not failed. I've just  | ['edison', 'failure', 'inspirational', 'paraphrased']
Eleanor Roosevelt | “A woman is like a tea bag; yo | ['misattributed-eleanor-roosevelt']
Steve Martin | “A day without sunshine is lik | ['humor', 'obvious', 'simile']


In [4]:
# 아래 두 개는 같은 일을 합니다
tags = [t.get_text(strip=True) for t in q.select('a.tag')]
 
tags = []
for t in q.select('a.tag'):
    tags.append(t.get_text(strip=True))

tags

['humor', 'obvious', 'simile']

In [5]:
# selector_test.py — 존재하지 않는 선택자를 써 봅니다 (마지막 줄에서 일부러 터집니다)
import requests
from bs4 import BeautifulSoup

res = requests.get('https://quotes.toscrape.com/', timeout=10)
soup = BeautifulSoup(res.text, 'html.parser')

print(soup.select('div.quotes'))          # 없는 클래스 → 빈 리스트 []
print(soup.select_one('span.txt'))        # 없는 클래스 → None
print(soup.select_one('span.txt').get_text())   # ← 여기서 터집니다

[]
None


AttributeError: 'NoneType' object has no attribute 'get_text'

In [6]:
# safe_pick.py — 없을 수 있는 값은 검사하고 넘어갑니다 (새 파일)
import requests
from bs4 import BeautifulSoup

res = requests.get('https://quotes.toscrape.com/', timeout=10)
soup = BeautifulSoup(res.text, 'html.parser')


def pick(el, selector, default=None):
    found = el.select_one(selector)
    return found.get_text(strip=True) if found else default

for q in soup.select('div.quote'):
    print(pick(q, 'small.author'), '|', pick(q, 'span.missing', '(없음)'))

Albert Einstein | (없음)
J.K. Rowling | (없음)
Albert Einstein | (없음)
Jane Austen | (없음)
Marilyn Monroe | (없음)
Albert Einstein | (없음)
André Gide | (없음)
Thomas A. Edison | (없음)
Eleanor Roosevelt | (없음)
Steve Martin | (없음)


In [7]:
# next_page.py — 페이지네이션 링크 확인 (새 파일)
import requests
from bs4 import BeautifulSoup

res = requests.get('https://quotes.toscrape.com/', timeout=10)
soup = BeautifulSoup(res.text, 'html.parser')

next_link = soup.select_one('li.next a')
print('next href:', next_link['href'] if next_link else None)

# 마지막 페이지에서는 None이 나옵니다 — 순회를 멈추는 신호가 됩니다
res_last = requests.get('https://quotes.toscrape.com/page/10/', timeout=10)
soup_last = BeautifulSoup(res_last.text, 'html.parser')
print('last page next:', soup_last.select_one('li.next a'))

next href: /page/2/
last page next: None


In [12]:
# crawler_pipeline.py — 수집(fetch) → 정제(parse) → 적재(load)
from __future__ import annotations

import time
from datetime import datetime

import pymysql
import requests
from bs4 import BeautifulSoup

BASE = 'https://quotes.toscrape.com'
HEADERS = {'User-Agent': 'data-eng-class-crawler/1.0 (learning purpose)'}
DELAY_SEC = 1.0  # 크롤링 예절: 요청 간격 1초

conn = pymysql.connect(host='127.0.0.1', port=3306, user='root',
                           password='dataeng123', database='shop', charset='utf8mb4')


# crawler_pipeline.py 의 def fetch … return res.text 를 이것으로 바꿉니다
def fetch(url: str, retries: int = 3) -> str:
    last_error = None
    for attempt in range(1, retries + 1):
        try:
            res = requests.get(url, headers=HEADERS, timeout=10)
            if res.status_code == 429:
                print(f'  429 too many requests — {attempt}회차, 5초 대기')
                time.sleep(5)
                continue
            res.raise_for_status()
            return res.text
        except requests.RequestException as e:
            last_error = e
            print(f'  요청 실패({attempt}/{retries}): {e}')
            time.sleep(2 * attempt)      # 갈수록 더 오래 쉰다
    raise RuntimeError(f'{url} 수집 실패') from last_error


def parse(html: str) -> tuple[list[dict], str | None]:
    soup = BeautifulSoup(html, 'html.parser')
    rows = []
    for q in soup.select('div.quote'):
        rows.append({
            'author': q.select_one('small.author').get_text(strip=True),
            'quote_text': q.select_one('span.text').get_text(strip=True),
            'tags': ','.join(t.get_text(strip=True) for t in q.select('a.tag')),
        })
    next_link = soup.select_one('li.next a')
    next_url = BASE + next_link['href'] if next_link else None
    return rows, next_url


def load(rows: list[dict]) -> int:
    saved = 0
    try:
        with conn.cursor() as cur:
            for r in rows:
                # INSERT IGNORE: UNIQUE 충돌(이미 수집한 명언)은 건너뜀
                saved += cur.execute(
                    'INSERT IGNORE INTO quotes (author, quote_text, tags, crawled_at) '
                    'VALUES (%s, %s, %s, %s)',
                    (r['author'], r['quote_text'], r['tags'], datetime.now()),
                )
        conn.commit()
    finally:
        conn.close()
    return saved


def run(max_pages: int = 10) -> None:
    url = BASE + '/'
    for page in range(1, max_pages + 1):
        html = fetch(url)
        rows, next_url = parse(html)
        saved = load(rows)
        print(f'page {page}: parsed {len(rows)}, saved {saved}')
        if not next_url:
            break
        url = next_url
        time.sleep(DELAY_SEC)


if __name__ == '__main__':
    run()

page 1: parsed 10, saved 0


Error: Already closed